# One-shot Document Classifier With DINO backbone(ODC) training

## Outline:
1. Import and define model 
2. Define Helper functions to loads images, normalize etc
3. Load the splitted dataset, apply collator and define the samplers
4. Training Loop
5. Evaluations
6. Save mode using torch script (hides the architecture, no need to worry if change architeture :) )


### Importing necessary libraries


In [5]:
import os
os.environ.setdefault("TRANSFORMERS_NO_TF", "1") # Ensure the flag
os.environ.setdefault("USE_TF", "0") # Ensure the flag
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True") # Helps reduce CUDA fragmentation
import csv
import json
import math
import random
import time
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as func
from torch.utils.data import BatchSampler, DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt

from transformers import AutoImageProcessor, Dinov2Model, get_linear_schedule_with_warmup


### Defining Paths


In [ ]:
SPLIT_OCR_ROOT = Path(r"A:\RealForm\processed\ocr_cache_splitted")
DEGRADED_ROOT = Path(r"A:\RealForm\processed\synthetic_fill_degraded")
ORIGINAL_ROOT = Path(r"A:\RealForm\processed\CommonFormsEnglish\images")
OUTPUT_DIR = Path(r"A:\RealForm\outputs\dino")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DINO_MODEL_NAME = "facebook/dinov2-base"
IMAGE_SIZE = {"height": 224, "width": 224}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cuda


### Defining helper functions


In [7]:
def read_json(path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)

@dataclass(frozen=True)
class DocSample:
    subset: str
    stem: str
    class_label: str
    image_path: Path
    record_id: int
    is_original: bool = False


### Loading the dataset


In [8]:
def build_original_image_index(original_root):
    index = {}
    for image_path in sorted(original_root.rglob("*"), key=lambda p: str(p).lower()):
        if image_path.is_file() and image_path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"}:
            index.setdefault(image_path.stem, image_path) # This will remove the extensition out of file name
    return index


def load_split(split_name):
    original_index = build_original_image_index(ORIGINAL_ROOT)
    query_docs = []
    original_docs = []

    split_dir = SPLIT_OCR_ROOT / split_name #Load the pre-splitted split
    template_dirs = sorted([p for p in split_dir.iterdir() if p.is_dir()], key=lambda p: p.name.lower())

    for template_dir in template_dirs:
        stem = template_dir.name
        original_image = original_index.get(stem)
        degraded_dir = DEGRADED_ROOT / stem

        if original_image is None or not degraded_dir.exists():
            continue

        original_docs.append( # Build docs sample
            DocSample(
                subset="original",
                stem=stem,
                class_label=stem,
                image_path=original_image.resolve(),
                record_id=0,
                is_original=True,
            )
        )

        # Keep the same split membership as the OCR split. DINOv2 does not use OCR, but the OCR files tell us which fills belong here.
        for record_id, image_path in enumerate(sorted(degraded_dir.glob("fill_*.png"), key=lambda p: p.name.lower()), start=1):
            ocr_path = template_dir / f"{image_path.stem}.ocr.json"
            if not ocr_path.exists():
                continue
            query_docs.append( # Build docs sample
                DocSample(
                    subset=split_name,
                    stem=stem,
                    class_label=stem,
                    image_path=image_path.resolve(),
                    record_id=record_id,
                    is_original=False,
                )
            )

    return query_docs, original_docs


train_query_docs, train_original_docs = load_split("train")
val_query_docs, val_original_docs = load_split("val")
test_query_docs, test_original_docs = load_split("test")

print("train", len(train_original_docs), len(train_query_docs))
print("val", len(val_original_docs), len(val_query_docs))
print("test", len(test_original_docs), len(test_query_docs))


train 8779 87790
val 285 2850
test 279 2790


### Define the model:
Model will comprise of 3 components

1. DINOv2 backbone (unfreeze every blocks)->> this will output embedding of 768 dim for every single visual tokens

2. Dense Retrieval Head (projection head) this will compress the embedding space from 768 to 128. Faster for retrival + compress informations so it only preserve necessary cues. This head will use contrastive loss

3. Arc Face Head is a classification head that will push interclass similarity away (sin) and intra-class similarity inward (cos). This will improve the margin of the embedding space, helping better retrieval!!!


In [9]:
class DenseRetriveHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(), # Use GELU (similar to the hidden activation of the transformers)
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 128), # Reduce to 128 dimension
        )

    def forward(self, pooled):
        return func.normalize(self.net(pooled), p=2, dim=-1)


class AccelerateArcFace(nn.Module): #ARCFACE is use to accelerate the seperation (contrastive loss) . This WILL BE REMOVE!!!!!
    def __init__(self, emb_dim, num_class):
        super().__init__()
        self.param = nn.Parameter(torch.empty(num_class, emb_dim))
        nn.init.xavier_uniform_(self.param) # initalize the weights
        # define the marign (cos + m) m ==> margin. Here we use margin of 0.2
        self.cos_margin = math.cos(0.2)
        self.sin_margin = math.sin(0.2)

    def forward(self, embed, labels):
        #Mormalize the embedding of the Dense head
        norm_emb = func.normalize(embed, p=2, dim=-1)
        norm_weight = func.normalize(self.param, p=2, dim=-1)
        #Compute the cosine similarity 
        cos = func.linear(norm_emb, norm_weight).clamp(-1.0 + 1e-7, 1.0 - 1e-7) # clamp so it the value does not vanish
        sin = torch.sqrt(torch.clamp(1.0 - cos * cos, min=1e-7)) # just pythagoras here
        phi = cos * self.cos_margin - sin * self.sin_margin # Cosine with angular margin applied: cos(theta + margin)
        one_hot = func.one_hot(labels, num_classes=self.param.shape[0]).to(dtype=cos.dtype)
        return (one_hot * phi + (1.0 - one_hot) * cos) * 30.0


def sup_con_loss(embs, labels, temperature=0.1):
    norm = func.normalize(embs, p=2, dim=-1)
    logits = norm @ norm.T / temperature #compute similarity scale by temp
    logits = logits - logits.max(dim=1, keepdim=True).values.detach() #solving numerical instability
    self_mask = torch.eye(labels.shape[0], device=labels.device, dtype=torch.bool) # this will mask the ground truht
    positive_mask = labels.unsqueeze(0).eq(labels.unsqueeze(1)) & ~self_mask # remove diag
    exp_logits = torch.exp(logits) * (~self_mask)
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True).clamp(min=1e-12)) # clamp to prevent numerical instability
    positive_counts = positive_mask.sum(dim=1)
    mean_log_prob = (positive_mask * log_prob).sum(dim=1) / positive_counts.clamp(min=1)
    valid = positive_counts > 0
    return embs.new_tensor(0.0) if not torch.any(valid) else -mean_log_prob[valid].mean()


In [ ]:
def find_dino_blocks(backbone):
    encoder = getattr(backbone, "encoder", None)
    if encoder is None:
        return []
    if hasattr(encoder, "layer"):
        return list(encoder.layer)
    if hasattr(encoder, "layers"):
        return list(encoder.layers)
    return []


class DinoODCModel(nn.Module):
    def __init__(self, num_class):
        super().__init__()
        self.backbone = Dinov2Model.from_pretrained(
            DINO_MODEL_NAME,
            use_safetensors=True,
        ) # backbone
        self.backbone.gradient_checkpointing_enable()
        hidden_size = self.backbone.config.hidden_size
        self.proj = DenseRetriveHead(hidden_size) # projection head
        self.accelerate = AccelerateArcFace(128, num_class) # accelerated arcface head
        # self.unfreeze_last_4_layers() ### TRIED FREEZE AND UNFREEZE
        self.optimizer = self.create_optimizer() # apply optimizer 

    def forward(self, pixel_values):
        # 1: compute embedding of backbone
        outputs = self.backbone(pixel_values=pixel_values, return_dict=True)
        #2 pass through projection head
        pooled = outputs.last_hidden_state[:, 0, :] #TAKING THE CLS TOKEN (first token0)
        return self.proj(pooled)

    def unfreeze_last_4_layers(self):
        for param in self.backbone.parameters():
            param.requires_grad = False # FREEZE ALL THE BACKBONE Weight

        blocks = find_dino_blocks(self.backbone)
        if blocks:
            for block in blocks[-4:]: # unfreeze last 4 layers
                for param in block.parameters():
                    param.requires_grad = True # unfreeze just in case
        else:
            print("WARNING: could not find DINOv2 encoder blocks; only projection head will train")

        # Final normalization can matter when only the tail blocks are trainable.
        for name, module in self.backbone.named_modules():
            if "layernorm" in name.lower() or name.lower().endswith("norm"):
                for param in module.parameters(recurse=False):
                    param.requires_grad = True # unfreeze just in case

        for param in self.proj.parameters():
            param.requires_grad = True # unfreeze just in case
        for param in self.accelerate.parameters():
            param.requires_grad = True # unfreeze just in case

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
        print(f"dino blocks found: {len(blocks)}; unfrozen last: {min(4, len(blocks))}")

    def create_optimizer(self):
        back_param = [p for p in self.backbone.parameters() if p.requires_grad]
        proj_param = [p for p in self.proj.parameters() if p.requires_grad]
        acc_param = [p for p in self.accelerate.parameters() if p.requires_grad]
        return torch.optim.AdamW(
            [
                {"params": back_param, "lr": 2e-5, "weight_decay": 0.01},
                {"params": proj_param, "lr": 1e-4, "weight_decay": 0.01},
                {"params": acc_param, "lr": 1e-4, "weight_decay": 0.01},
            ]
        )


### Defining the collator to apply the processor of DINOv2 and defining helper class for querying


In [11]:
@dataclass(frozen=True)
class TemplateExample:
    document: DocSample
    class_label: str
    train_label_id: int


class TemplateDocumentDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


class DinoCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        images = [Image.open(ex.document.image_path).convert("RGB") for ex in examples]
        try:
            encoding = self.processor(images=images, return_tensors="pt")
        finally:
            for img in images:
                img.close()
        return {
            "pixel_values": encoding["pixel_values"],
            "labels": torch.tensor([ex.train_label_id for ex in examples], dtype=torch.long),
            "documents": [ex.document for ex in examples],
        }


processor = AutoImageProcessor.from_pretrained(DINO_MODEL_NAME, size=IMAGE_SIZE)
collator = DinoCollator(processor)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


### Batch balancer to ensure that every batch contains equal amount of sample per batch!


In [12]:
class BalanceBatchSampler(BatchSampler):
    def __init__(self, examples, classes_per_batch=4, samples_per_class=2, batches_per_epoch=None, seed=42):
        self.examples = examples
        self.classes_per_batch = classes_per_batch
        self.samples_per_class = samples_per_class
        self.seed = seed
        self.class_to_indices = {}
        for idx, ex in enumerate(examples): #map the classes to their indicies eg {1:[2,3], 2:[0,4]}
            self.class_to_indices.setdefault(ex.train_label_id, []).append(idx)
        self.class_ids = sorted(self.class_to_indices.keys())
        default_batches = math.ceil(len(examples) / max(1, classes_per_batch * samples_per_class))
        self.batches_per_epoch = batches_per_epoch or default_batches

    def __len__(self):
        return self.batches_per_epoch

    def __iter__(self):
        rng = random.Random(self.seed + int(time.time()))
        for _ in range(self.batches_per_epoch):
            chosen_classes = rng.sample(self.class_ids, k=min(self.classes_per_batch, len(self.class_ids)))
            batch_indices = []
            for class_id in chosen_classes:
                candidates = self.class_to_indices[class_id]
                if len(candidates) >= self.samples_per_class:
                    batch_indices.extend(rng.sample(candidates, k=self.samples_per_class))
                else:
                    batch_indices.extend(rng.choices(candidates, k=self.samples_per_class))
            yield batch_indices # each batch have equal amount sample per class ==> so that it can perform supcon


def build_examples(documents, class_to_id):
    return [
        TemplateExample(document=doc, class_label=doc.class_label, train_label_id=class_to_id[doc.class_label])
        for doc in documents
    ]


def build_inference_examples(documents):
    labels = sorted({doc.class_label for doc in documents})
    class_to_id = {label: idx for idx, label in enumerate(labels)}
    return build_examples(documents, class_to_id)


train_labels = sorted({doc.class_label for doc in train_query_docs})
train_class_to_id = {label: idx for idx, label in enumerate(train_labels)}
train_examples = build_examples(train_query_docs, train_class_to_id)
train_dataset = TemplateDocumentDataset(train_examples)

train_sampler = BalanceBatchSampler(
    examples=train_examples,
    classes_per_batch=4,
    samples_per_class=2,
    batches_per_epoch=None,
    seed=42,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    num_workers=0,
    collate_fn=collator,
)

print("train labels/templates:", len(train_labels))
print("train examples:", len(train_examples))
print("train batches:", len(train_loader))


train labels/templates: 8779
train examples: 87790
train batches: 10974


BUILD THE MODEL!


In [13]:
EPOCHS = 10
model = DinoODCModel(num_class=len(train_labels)).to(DEVICE)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    model.optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

print("total_steps:", total_steps)
print("warmup_steps:", warmup_steps)

total_steps: 109740
warmup_steps: 10974


c:\Users\thanh\anaconda3\envs\LayoutLM\lib\site-packages\torch\nn\modules\module.py:1326: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


### TRAINING LOOP

In [14]:
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_arc = 0.0
    total_supcon = 0.0

    for step, batch in enumerate(train_loader, start=1):
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        model.optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"): # automatic precision
            embs = model(pixel_values=pixel_values)
            arc_logits = model.accelerate(embs, labels)
            arc_loss = func.cross_entropy(arc_logits, labels)
            contrast_loss = sup_con_loss(embs, labels)
            loss = 0.2 * arc_loss + 0.8 * contrast_loss

        scaler.scale(loss).backward()
        scaler.unscale_(model.optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(model.optimizer)
        scaler.update()
        scheduler.step()

        total_loss += float(loss.item())
        total_arc += float(arc_loss.item())
        total_supcon += float(contrast_loss.item())

        if step == 1 or step % 20 == 0 or step == len(train_loader):
            print(
                f"epoch {epoch} step {step}/{len(train_loader)} "
                f"loss={loss.item():.4f} arc={arc_loss.item():.4f} supcon={contrast_loss.item():.4f}"
            )

    row = {
        "epoch": epoch,
        "train_loss": total_loss / len(train_loader),
        "train_arc_loss": total_arc / len(train_loader),
        "train_supcon_loss": total_supcon / len(train_loader),
    }
    history.append(row)
    print("epoch complete:", row)

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": model.optimizer.state_dict(),
            "history": history,
            "model_name": DINO_MODEL_NAME,
            "pooling": "cls_token",
        },
        OUTPUT_DIR / "last_dino_projection_model.pt",
    )


c:\Users\thanh\anaconda3\envs\LayoutLM\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


epoch 1 step 1/10974 loss=5.1208 arc=19.7061 supcon=1.4744
epoch 1 step 20/10974 loss=5.1579 arc=17.8555 supcon=1.9835
epoch 1 step 40/10974 loss=5.0984 arc=20.2920 supcon=1.3000
epoch 1 step 60/10974 loss=4.2555 arc=15.5977 supcon=1.4199
epoch 1 step 80/10974 loss=4.6457 arc=16.9053 supcon=1.5808
epoch 1 step 100/10974 loss=4.9881 arc=18.2939 supcon=1.6617
epoch 1 step 120/10974 loss=4.9188 arc=18.5547 supcon=1.5098
epoch 1 step 140/10974 loss=5.0807 arc=19.6465 supcon=1.4393
epoch 1 step 160/10974 loss=4.3771 arc=19.1797 supcon=0.6764
epoch 1 step 180/10974 loss=4.0899 arc=16.7627 supcon=0.9217
epoch 1 step 200/10974 loss=3.8119 arc=17.2324 supcon=0.4567
epoch 1 step 220/10974 loss=4.3560 arc=18.0645 supcon=0.9289
epoch 1 step 240/10974 loss=4.4840 arc=18.3770 supcon=1.0108
epoch 1 step 260/10974 loss=5.2334 arc=19.1406 supcon=1.7567
epoch 1 step 280/10974 loss=4.0306 arc=15.8672 supcon=1.0715
epoch 1 step 300/10974 loss=4.9605 arc=20.0488 supcon=1.1884
epoch 1 step 320/10974 loss=4.

## Evaluations


In [15]:
import gc, torch
gc.collect() # Reclaims Python memory
torch.cuda.empty_cache() # Releases unused cached GPU memory
torch.cuda.empty_cache()

### Defining helper function to evaluate the query of samples with its original templates


In [16]:
def build_inference_loader(query_docs, original_docs, batch_size=8):
    docs = query_docs + original_docs
    examples = build_inference_examples(docs)
    return DataLoader(
        TemplateDocumentDataset(examples),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=collator,
    )


def collect_embeddings(model, loader):
    model.eval()
    all_embs = [] # stores all the embeddings
    all_docs = [] # stores all the DocSample
    with torch.no_grad():
        for batch in loader:
            pixel_values = batch["pixel_values"].to(DEVICE)
            embs = model(pixel_values=pixel_values)
            all_embs.append(embs.detach().cpu().numpy())
            all_docs.extend(batch["documents"])
    return np.concatenate(all_embs, axis=0), all_docs # concate to a single matrix


def retrieval_vs_original(query_docs, query_embs, original_docs, original_embs):
    query_embs = query_embs / np.maximum(np.linalg.norm(query_embs, axis=1, keepdims=True), 1e-12) # prevent divide 0 and normalzie
    original_embs = original_embs / np.maximum(np.linalg.norm(original_embs, axis=1, keepdims=True), 1e-12) # prevent divide 0 and normalzie
    original_labels = np.array([doc.stem for doc in original_docs])

    top1 = top5 = top10 = 0
    reciprocal_ranks = []
    for i, query_doc in enumerate(query_docs):
        scores = query_embs[i] @ original_embs.T # cosine similarity
        order = np.argsort(-scores) # rerdering from highest to lowest 
        ranked_labels = original_labels[order] # apply the order to the label  
        rank = np.where(ranked_labels == query_doc.stem)[0][0] + 1 # position of the original doc
        top1 += int(rank == 1)
        top5 += int(rank <= 5)
        top10 += int(rank <= 10)
        reciprocal_ranks.append(1.0 / rank) # for calculating mrr (reciprocal rank higher = better)

    n = len(query_docs)
    return {
        "query_count": n,
        "gallery_count": len(original_docs),
        "retrieval_at_1": top1 / n,
        "retrieval_at_5": top5 / n,
        "retrieval_at_10": top10 / n,
        "mrr": float(sum(reciprocal_ranks) / n),
        "correct_top1": top1,
    }


def evaluate_split(query_docs, original_docs, name):
    loader = build_inference_loader(query_docs, original_docs, batch_size=8)
    embs, docs = collect_embeddings(model, loader)
    query_count = len(query_docs)
    original_count = len(original_docs)
    query_embs = embs[:query_count]
    original_embs = embs[query_count:query_count + original_count]
    metrics = retrieval_vs_original(query_docs, query_embs, original_docs, original_embs)
    print(name, metrics)
    write_json(OUTPUT_DIR / f"{name}_retrieval_metrics.json", metrics)
    return metrics


In [17]:
val_metrics = evaluate_split(
    query_docs=val_query_docs,
    original_docs=val_original_docs,
    name="val",
)

val {'query_count': 2850, 'gallery_count': 285, 'retrieval_at_1': 0.8631578947368421, 'retrieval_at_5': 0.9477192982456141, 'retrieval_at_10': 0.963859649122807, 'mrr': 0.9018726619693443, 'correct_top1': 2460}


In [18]:
test_metrics = evaluate_split(
    query_docs=test_query_docs,
    original_docs=test_original_docs,
    name="test",
)

test {'query_count': 2790, 'gallery_count': 279, 'retrieval_at_1': 0.8551971326164874, 'retrieval_at_5': 0.953763440860215, 'retrieval_at_10': 0.9684587813620071, 'mrr': 0.8997464794797696, 'correct_top1': 2386}


### Save the model


In [19]:
FINAL_MODEL_PATH = OUTPUT_DIR / "dino_projection_scripted.pt"

class DinoModelWrap(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.backbone = trained_model.backbone
        self.proj = trained_model.proj

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values, return_dict=True)
        pooled = outputs.last_hidden_state[:, 0, :]
        return self.proj(pooled)


model.eval()
model.cpu()
wrapped_model = DinoModelWrap(model).eval().cpu()
example_pixel_values = torch.zeros((1, 3, IMAGE_SIZE["height"], IMAGE_SIZE["width"]), dtype=torch.float32)

with torch.no_grad():
    traced = torch.jit.trace(wrapped_model, (example_pixel_values,), strict=False)

FINAL_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
traced.save(str(FINAL_MODEL_PATH))
print("saved", FINAL_MODEL_PATH)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
c:\Users\thanh\anaconda3\envs\LayoutLM\lib\site-packages\transformers\models\dinov2\modeling_dinov2.py:143: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self.num_channels:


saved A:\RealForm\outputs\dino\dino_projection_scripted.pt


In [20]:
from pathlib import Path
from torch.utils.data import DataLoader

EXTRA8K_DEGRADED_ROOT = Path(r"A:\RealForm\processed\synthetic_fill_images_8k_extra_3fills_degraded")
EXTRA8K_OCR_ROOT = Path(r"A:\RealForm\processed\synthetic_fill_images_8k_extra_3fills_degraded_ocr_clean\ocr_json")
ORIGINAL_IMAGE_ROOT = Path(r"A:\RealForm\processed\CommonFormsEnglish\images")

def build_original_image_index(original_root):
    index = {}
    for image_path in sorted(original_root.rglob("*"), key=lambda p: str(p).lower()):
        if image_path.is_file() and image_path.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"}:
            index.setdefault(image_path.stem, image_path.resolve())
    return index

original_index = build_original_image_index(ORIGINAL_IMAGE_ROOT)

query_docs_8k = []
original_docs_8k = []

template_dirs = sorted([p for p in EXTRA8K_OCR_ROOT.iterdir() if p.is_dir()], key=lambda p: p.name.lower())

for template_dir in template_dirs:
    stem = template_dir.name
    original_image = original_index.get(stem)
    degraded_dir = EXTRA8K_DEGRADED_ROOT / stem

    if original_image is None or not degraded_dir.exists():
        continue

    original_docs_8k.append(
        DocSample(
            subset="extra8k_original",
            stem=stem,
            class_label=stem,
            image_path=original_image,
            record_id=0,
            is_original=True,
        )
    )

    for record_id, image_path in enumerate(sorted(degraded_dir.glob("fill_*.png"), key=lambda p: p.name.lower()), start=1):
        ocr_path = template_dir / f"{image_path.stem}.ocr.json"
        if not ocr_path.exists():
            continue
        query_docs_8k.append(
            DocSample(
                subset="extra8k_query",
                stem=stem,
                class_label=stem,
                image_path=image_path.resolve(),
                record_id=record_id,
                is_original=False,
            )
        )

print("gallery templates:", len(original_docs_8k))
print("query images:", len(query_docs_8k))

def build_eval_loader(query_docs, original_docs, batch_size=8):
    docs = query_docs + original_docs
    examples = build_inference_examples(docs)
    return DataLoader(
        TemplateDocumentDataset(examples),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=collator,
    )

loader_8k = build_eval_loader(query_docs_8k, original_docs_8k, batch_size=8)
embs_8k, _ = collect_embeddings(model, loader_8k)

query_count = len(query_docs_8k)
gallery_count = len(original_docs_8k)

query_embs_8k = embs_8k[:query_count]
gallery_embs_8k = embs_8k[query_count:query_count + gallery_count]

metrics_8k = retrieval_vs_original(
    query_docs=query_docs_8k,
    query_embs=query_embs_8k,
    original_docs=original_docs_8k,
    original_embs=gallery_embs_8k,
)

print(metrics_8k)

gallery templates: 8000
query images: 24000
{'query_count': 24000, 'gallery_count': 8000, 'retrieval_at_1': 0.39175, 'retrieval_at_5': 0.5833333333333334, 'retrieval_at_10': 0.6572916666666667, 'mrr': 0.48314686012268065, 'correct_top1': 9402}
